In [1]:
%matplotlib inline

Tensors
=======

Tensors are a specialized data structure that are very similar to arrays
and matrices. In PyTorch, we use tensors to encode the inputs and
outputs of a model, as well as the model's parameters.

Tensors are similar to [NumPy's](https://numpy.org/) ndarrays, except
that tensors can run on GPUs or other hardware accelerators. In fact,
tensors and NumPy arrays can often share the same underlying memory,
eliminating the need to copy data (see
`bridge-to-np-label`{.interpreted-text role="ref"}). Tensors are also
optimized for automatic differentiation (we\'ll see more about that
later in the [Autograd](autogradqs_tutorial.html) section). If you're
familiar with ndarrays, you'll be right at home with the Tensor API. If
not, follow along!


In [2]:
import torch
import numpy as np

Initializing a Tensor
=====================
**Directly from data**

Tensors can be created directly from data. The data type is
automatically inferred.


In [4]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)
x_data

tensor([[1, 2],
        [3, 4]])

**From a NumPy array**

Tensors can be created from NumPy arrays (and vice versa - see
`bridge-to-np-label`{.interpreted-text role="ref"}).


In [6]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
x_np

tensor([[1, 2],
        [3, 4]])

**From another tensor:**

The new tensor retains the properties (shape, datatype) of the argument
tensor, unless explicitly overridden.


In [7]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.9070, 0.7754],
        [0.2293, 0.2008]]) 



**With random or constant values:**

`shape` is a tuple of tensor dimensions. In the functions below, it
determines the dimensionality of the output tensor.


In [8]:
shape = (2,3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.1362, 0.9136, 0.6235],
        [0.2337, 0.4913, 0.8720]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


------------------------------------------------------------------------


Attributes of a Tensor
======================

Tensor attributes describe their shape, datatype, and the device on
which they are stored.


In [9]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


------------------------------------------------------------------------


Operations on Tensors
=====================

Over 1200 tensor operations, including arithmetic, linear algebra,
matrix manipulation (transposing, indexing, slicing), sampling and more
are comprehensively described
[here](https://pytorch.org/docs/stable/torch.html).

Each of these operations can be run on the CPU and
[Accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If you're using Colab, allocate an
accelerator by going to Runtime \> Change runtime type \> GPU.

By default, tensors are created on the CPU. We need to explicitly move
tensors to the accelerator using `.to` method (after checking for
accelerator availability). Keep in mind that copying large tensors
across devices can be expensive in terms of time and memory!


In [10]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

Try out some of the operations from the list. If you\'re familiar with
the NumPy API, you\'ll find the Tensor API a breeze to use.


**Standard numpy-like indexing and slicing:**


In [11]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


**Joining tensors** You can use `torch.cat` to concatenate a sequence of
tensors along a given dimension. See also
[torch.stack](https://pytorch.org/docs/stable/generated/torch.stack.html),
another tensor joining operator that is subtly different from
`torch.cat`.


In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

**Arithmetic operations**


In [12]:
# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)


# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

**Single-element tensors** If you have a one-element tensor, for example
by aggregating all values of a tensor into one value, you can convert it
to a Python numerical value using `item()`:


In [13]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

12.0 <class 'float'>


**In-place operations** Operations that store the result into the
operand are called in-place. They are denoted by a `_` suffix. For
example: `x.copy_(y)`, `x.t_()`, will change `x`.


In [14]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>In-place operations save some memory, but can be problematic when computing derivatives because of an immediate lossof history. Hence, their use is discouraged.</p>

</div>



------------------------------------------------------------------------


Bridge with NumPy {#bridge-to-np-label}
=================

Tensors on the CPU and NumPy arrays can share their underlying memory
locations, and changing one will change the other.


Tensor to NumPy array
=====================


In [15]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


A change in the tensor reflects in the NumPy array.


In [16]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


NumPy array to Tensor
=====================


In [18]:
n = np.ones(5)
t = torch.from_numpy(n)
t

tensor([1., 1., 1., 1., 1.], dtype=torch.float64)

Changes in the NumPy array reflects in the tensor.


In [19]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]


In [20]:
import torch
print(torch.__version__)

2.11.0+cu128


In [22]:
# device availabilty
if torch.cuda.is_available():
  print("GPU is available..✅")
  print(f"using GPU:{torch.cuda.get_device_name(0)}")
else:
  print("using cpu..✅")

GPU is available..✅
using GPU:Tesla T4


### ***Crating Tensors***

In [24]:
# using empty
a = torch.empty(2,3)
print(a)
type(a)

tensor([[1.9641e+37, 3.4034e-18, 7.0976e+22],
        [4.3605e+27, 1.5766e-19, 3.0881e+29]])


torch.Tensor

In [26]:
torch.zeros(3,3)

tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])

In [27]:
torch.ones(3,2)

tensor([[1., 1.],
        [1., 1.],
        [1., 1.]])

In [28]:
torch.rand(3,3)

tensor([[0.7848, 0.6219, 0.8167],
        [0.1616, 0.5857, 0.1579],
        [0.7640, 0.6614, 0.5272]])

In [29]:
# manual seed
torch.manual_seed(42)
torch.rand(3,3)

tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408]])

In [32]:
# other ways to create tensors

a = torch.arange(0,20,2)
print(a)

space = torch.linspace(0,10,3)
print(space)

identity = torch.eye(4)
print(identity)

full = torch.full((3,3),8)
print(full)

tensor([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18])
tensor([ 0.,  5., 10.])
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])
tensor([[8, 8, 8],
        [8, 8, 8],
        [8, 8, 8]])


In [33]:
X = torch.tensor([[1,2,3],[4,5,6]])
print(X)
print(X.shape)

tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])


In [35]:
torch.zeros_like(X)

tensor([[0, 0, 0],
        [0, 0, 0]])

In [37]:
torch.ones_like(X)

tensor([[1, 1, 1],
        [1, 1, 1]])

In [39]:
torch.empty_like(X)

tensor([[133352269294976,      1270596624,               0],
        [134492605906944,               0,               0]])

In [41]:
torch.rand_like(X,dtype=torch.float64)

tensor([[0.0655, 0.7486, 0.5417],
        [0.4352, 0.5912, 0.0196]], dtype=torch.float64)

In [42]:
torch.tensor([1.0,2.0,3.0], dtype=torch.int32)

tensor([1, 2, 3], dtype=torch.int32)

### ***Mathematical Operation***

In [43]:
# scaler operation
x = torch.rand(3,3)
x

tensor([[0.4414, 0.2969, 0.8317],
        [0.1053, 0.2695, 0.3588],
        [0.1994, 0.5472, 0.0062]])

In [44]:
# addition
x + 2
# substraction
x - 2
# multiplication
x * 3
# division
x / 3
# int division
(x * 100)//3
# mod
((x * 100)//3)%2
# power
x**2

tensor([[1.9480e-01, 8.8162e-02, 6.9170e-01],
        [1.1091e-02, 7.2627e-02, 1.2875e-01],
        [3.9746e-02, 2.9942e-01, 3.7951e-05]])

In [46]:
c = torch.tensor([1.3, -2, 3.8, -4])
torch.abs(c)

tensor([1.3000, 2.0000, 3.8000, 4.0000])

In [47]:
torch.round(c)

tensor([ 1., -2.,  4., -4.])

In [48]:
d = torch.tensor([1.9, 2.3, 3.7, 4.4])

c = torch.ceil(d)
print(c)

f = torch.floor(d)
print(f)

clamp = torch.clamp(d,min=2,max=4)
print(clamp)

tensor([2., 3., 4., 5.])
tensor([1., 2., 3., 4.])
tensor([2.0000, 2.3000, 3.7000, 4.0000])


In [49]:
e = torch.randint(size=(3,3), low=0, high=10, dtype=torch.float32)
e

tensor([[2., 0., 5.],
        [9., 3., 4.],
        [9., 6., 2.]])

### ***Tenor Special Function***

In [50]:
k = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float32)
k

tensor([[0., 6., 2.],
        [7., 9., 7.]])

In [51]:
torch.log(k)

tensor([[  -inf, 1.7918, 0.6931],
        [1.9459, 2.1972, 1.9459]])

In [52]:
torch.exp(k)

tensor([[1.0000e+00, 4.0343e+02, 7.3891e+00],
        [1.0966e+03, 8.1031e+03, 1.0966e+03]])

In [53]:
torch.sqrt(k)

tensor([[0.0000, 2.4495, 1.4142],
        [2.6458, 3.0000, 2.6458]])

In [54]:
torch.sigmoid(k)

tensor([[0.5000, 0.9975, 0.8808],
        [0.9991, 0.9999, 0.9991]])

In [55]:
torch.softmax(k, dim=0)

tensor([[9.1105e-04, 4.7426e-02, 6.6929e-03],
        [9.9909e-01, 9.5257e-01, 9.9331e-01]])

In [56]:
torch.relu(k)

tensor([[0., 6., 2.],
        [7., 9., 7.]])

In [57]:
m = torch.rand(2,3)
n = torch.rand(2,3)

# apply operations
add = m.add_(n)
print(add)

tensor([[0.5882, 0.9051, 1.4997],
        [1.3979, 0.9024, 0.6632]])


In [58]:
# copying tensora
copy = torch.rand(3,3)
copy

tensor([[0.4913, 0.8913, 0.1447],
        [0.5315, 0.1587, 0.6542],
        [0.3278, 0.6532, 0.3958]])

In [59]:
orig = copy
orig

tensor([[0.4913, 0.8913, 0.1447],
        [0.5315, 0.1587, 0.6542],
        [0.3278, 0.6532, 0.3958]])

In [60]:
id(copy)

133346270564816

In [61]:
id(orig)

133346270564816